<a href="https://colab.research.google.com/github/Jake-Pepaj/TV-Ratings-Predictor/blob/main/Data450project_Henriquez_Pepaj.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Link to Data: https://www.kaggle.com/datasets/dbdmobile/myanimelist-dataset?select=anime-dataset-2023.csv

Business Impact References
https://www.animenewsnetwork.com/news/2023-12-18/aja-anime-industry-grew-by-6.8-percent-in-2022/.205649

https://www.cbr.com/2026-anime-streaming-trio-netflix-crunchyroll-prime/





Data Exploration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("anime-dataset-2023.csv")

print("DATASET DIMENSIONS & DATA TYPES")
print(f"Total Rows:    {df.shape[0]:,}")
print(f"Total Columns: {df.shape[1]}")
print("\nColumn Data Types:")
print(df.dtypes)

# Clean 'Score' to numeric to identify true distributions and invalid placeholders
df['Score_num'] = pd.to_numeric(df['Score'], errors='coerce')

print("\n Target Variable ('Score') Summary")
print(df['Score_num'].describe().round(2))
missing_target = df['Score_num'].isna().sum()
print(f"Missing/Unrated Scores: {missing_target:,} ({missing_target / len(df) * 100:.2f}%)")

DATASET DIMENSIONS & DATA TYPES
Total Rows:    24,905
Total Columns: 24

Column Data Types:
anime_id         int64
Name            object
English name    object
Other name      object
Score           object
Genres          object
Synopsis        object
Type            object
Episodes        object
Aired           object
Premiered       object
Status          object
Producers       object
Licensors       object
Studios         object
Source          object
Duration        object
Rating          object
Rank            object
Popularity       int64
Favorites        int64
Scored By       object
Members          int64
Image URL       object
dtype: object

 Target Variable ('Score') Summary
count    15692.00
mean         6.38
std          0.93
min          1.85
25%          5.73
50%          6.39
75%          7.06
max          9.10
Name: Score_num, dtype: float64
Missing/Unrated Scores: 9,213 (36.99%)


In [ ]:
#("Most Common Anime Types")
type_counts = df['Type'].value_counts()
print(type_counts)
#Wanted to see how many entries were each type of format​
print("\n Average Score By Anime Type")
type_score_summary = (
    df.groupby('Type')['Score_num']
    .agg(Count='count', Mean_Score='mean', Median_Score='median', Std_Dev='std')
    .sort_values(by='Mean_Score', ascending=False)
    .round(2)
)
print(type_score_summary)

In [ ]:
print("\n Most Common Anime Sources")
source_counts = df['Source'].value_counts()
print(source_counts.head(10))

print("\n Average Score By Source Material")
source_score_summary = (
    df.groupby('Source')['Score_num']
    .agg(Count='count', Mean_Score='mean', Median_Score='median', Std_Dev='std')
    .sort_values(by='Mean_Score', ascending=False)
    .round(2)
)
print(source_score_summary.head(10))

In [ ]:
# Checking which values are unknown and how much of our data is missing
#Good to see how much of our data needs to be dropped before creating the test/train set
print("\n MISSING & 'UNKNOWN'")
unknown_counts = (df == 'UNKNOWN').sum() + df.isna().sum()
unknown_pct = (unknown_counts / len(df)) * 100
top_missing = unknown_pct[unknown_pct > 0].sort_values(ascending=False).head(8)
print(top_missing.round(2).astype(str) + "%")

In [ ]:
#Plot 1: Distribution of Target Variable (Score)
#We wanted a physical representation of our target value
#This graph shows that anime scores are roughly bell-shaped/normally distributed with a slight leftskew, centered near the plotted median line
#The target variable is slightly left-skewed with a median of roughly 6.5. Very few shows score below 4.0 or above 8.5

plt.figure(figsize=(8, 4.5))
sns.histplot(df['Score_num'].dropna(), bins=35, kde=True)
median_val = df['Score_num'].median()
plt.axvline(median_val, linestyle='--', linewidth=2, label=f'Median: {median_val:.2f}')
plt.title("Distribution of Anime Scores (Target Variable)", fontsize=13, weight='bold')
plt.xlabel("Score (Scale 1.0 - 10.0)")
plt.ylabel("Count of Anime Titles")
plt.legend()
plt.show()

In [ ]:
#Plot 2: Score Distribution Across Anime Types
format_list = ['TV', 'Movie', 'OVA', 'ONA', 'Special', 'Music']
filtered_types = df[df['Type'].isin(format_list) & df['Score_num'].notna()]

plt.figure(figsize=(9, 4.5))
type_order = filtered_types.groupby('Type')['Score_num'].median().sort_values(ascending=False).index
sns.boxplot(data=filtered_types, x='Type', y='Score_num', order=type_order)
plt.title("Score Distribution Across Anime Format Types", fontsize=13, weight='bold')
plt.xlabel("Format Type")
plt.ylabel("Score")
plt.show()

Data Preparation

In [ ]:
import pandas as pd
data = pd.read_csv("anime.csv")

In [ ]:
data.columns

Index(['anime_id', 'Name', 'English name', 'Other name', 'Score', 'Genres',
       'Synopsis', 'Type', 'Episodes', 'Aired', 'Premiered', 'Status',
       'Producers', 'Licensors', 'Studios', 'Source', 'Duration', 'Rating',
       'Rank', 'Popularity', 'Favorites', 'Scored By', 'Members', 'Image URL'],
      dtype='object')

In [ ]:
# Drop unimportant columns
columns_to_keep = [
    'Name',
    'Score',
    'Genres',
    'Type',
    'Studios',
    'Licensors',
    'Producers',
    'Source',
    'Rating',
]

data = data[columns_to_keep]

print(data.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24905 entries, 0 to 24904
Data columns (total 9 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Name       24905 non-null  object
 1   Score      24905 non-null  object
 2   Genres     24905 non-null  object
 3   Type       24905 non-null  object
 4   Studios    24905 non-null  object
 5   Licensors  24905 non-null  object
 6   Producers  24905 non-null  object
 7   Source     24905 non-null  object
 8   Rating     24905 non-null  object
dtypes: object(9)
memory usage: 1.7+ MB
None


In [ ]:
# Clean Type: Drop music and unknown
data['Type'].unique()

data.drop(data[data['Type'] == 'Music'].index, inplace=True)
data.drop(data[data['Type'] == 'UNKNOWN'].index, inplace=True)

In [ ]:
data.shape

(22145, 9)

In [ ]:
#Only Studios with more than 40 values
data['Studios'].unique()
SC = data['Studios'].value_counts()
SC[SC > 40]
data2 = data[data.groupby('Studios')['Studios'].transform('count') >= 40]
data2.drop(data2[data2['Studios'] == 'UNKNOWN'].index, inplace=True)

data2['Studios'].value_counts()


/tmp/ipykernel_1911/3308427351.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data2.drop(data2[data2['Studios'] == 'UNKNOWN'].index, inplace=True)


,count
Studios,
Toei Animation,829
Sunrise,511
J.C.Staff,383
Shanghai Animation Film Studio,335
Madhouse,324
...,...
White Fox,41
Polygon Pictures,41
Fanworks,41


In [ ]:
#Only licensors with more than 20 values
data2 = data2[data2.groupby('Licensors')['Licensors'].transform('count') >= 20]
data2.drop(data2[data2['Licensors'] == 'UNKNOWN'].index, inplace=True)
data2['Licensors'].value_counts()

,count
Licensors,
Funimation,662
Sentai Filmworks,533
Discotek Media,202
Aniplex of America,176
VIZ Media,128
Media Blasters,117
ADV Films,82
Nozomi Entertainment,60
Kitty Media,53


In [ ]:
data2.shape

(2541, 9)

In [ ]:
#Creating new column "series" to be able to group shows and movies from the same series
import re

# Function to grab the core text before common sequel delimiters
def extract_franchise(title):
    # Splits the title at ':', '-', ' Season', ' Part', or ' 2nd' and keeps the first chunk
    base_name = re.split(r':|-| Season| Part| \d(?:st|nd|rd|th)', str(title))[0]
    return base_name.strip()

# Create the new column
data2['Series'] = data2['Name'].apply(extract_franchise)

# Check your work to see the groupings
print(data2['Series'].value_counts().head(5))

Series
Lupin III           20
Code Geass          13
Naruto              11
Sword Art Online     8
Koukaku Kidoutai     7
Name: count, dtype: int64


In [ ]:
aot_count = data2['Series'].value_counts().get('Shingeki no Kyojin', 0)
print(f"Total entries for this series: {aot_count}")

Total entries for this series: 7


In [ ]:


#Clean Genres
genres_to_drop = ['Award Winning', 'UNKNOWN', 'Unknown']

def clean_genres(genre_string):
    # Handle any empty rows to prevent errors
    if pd.isna(genre_string):
        return genre_string

    # Split the string by comma and remove any accidental extra spaces
    genre_list = [g.strip() for g in genre_string.split(',')]

    # Keep the genre only if it is NOT in our drop list
    cleaned_list = [g for g in genre_list if g not in genres_to_drop]

    # Rejoin the surviving genres back into a clean string
    return ", ".join(cleaned_list)

# Apply the cleaning function to the column
data2['Genres'] = data2['Genres'].apply(clean_genres)


print(data2['Genres'].head(5))

0                           Action, Sci-Fi
2                Action, Adventure, Sci-Fi
3     Action, Drama, Mystery, Supernatural
9                 Drama, Mystery, Suspense
10              Action, Adventure, Fantasy
Name: Genres, dtype: object


In [ ]:
#Drop unknown sources, as well as music media sources
data2['Source'].value_counts()
data2.drop(data2[data2['Source'] == 'Unknown'].index, inplace=True)
data2.drop(data2[data2['Source'] == 'Radio'].index, inplace=True)
data2.drop(data2[data2['Source'] == 'Music'].index, inplace=True)

In [ ]:
data2.shape

(2426, 10)

In [ ]:
data2['Rating'].unique()

array(['R - 17+ (violence & profanity)', 'PG-13 - Teens 13 or older',
       'R+ - Mild Nudity', 'G - All Ages', 'PG - Children', 'Rx - Hentai',
       'UNKNOWN'], dtype=object)

In [ ]:
#Clean rating

data2.drop(data2[data2['Rating'] == 'Rx - Hentai'].index, inplace=True)
data2.drop(data2[data2['Rating'] == 'UNKNOWN'].index, inplace=True)

# Clean Rating names
data2['Rating'] = data2['Rating'].str.split(' - ').str[0].str.strip()


data2['Rating'].value_counts()

,count
Rating,
PG-13,1482
R,431
R+,247
G,84
PG,65


In [ ]:
data2.shape

(2309, 10)

In [ ]:
#Make sure unknown scores are dropped
data2.drop(data2[data2['Score'] == 'UNKNOWN'].index, inplace=True)

In [ ]:
#Drop Name, not needed for predictions
data2 = data2.drop('Name', axis=1)

In [ ]:
data2.shape

(2306, 9)

In [ ]:
# List the exact columns you want to one-hot encode
columns_to_encode = ['Type', 'Source', 'Rating','Licensors', 'Studios' ]

# Apply get_dummies and make sure data is 1s and 0s
data2 = pd.get_dummies(data2, columns=columns_to_encode, drop_first=True, dtype=int)

data2.shape

(2306, 105)

In [ ]:
# Separate genres using the , to get singular genres
genre_dummies = data2['Genres'].str.get_dummies(', ')

# Merge the new binary columns back into the main dataframe
data2 = pd.concat([data2, genre_dummies], axis=1)

# Drop the original text column since it's no longer needed for modeling
data2 = data2.drop('Genres', axis=1)


data2.shape

(2306, 123)

In [ ]:
# Too many Series' to one hot encode, so take the size of each series instead
data2['Series_Size'] = data2.groupby('Series')['Series'].transform('count')

# Drop the text column
data2 = data2.drop('Series', axis=1)

data2.shape

(2306, 123)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

#Separate features and target variable
X = data2.drop('Score', axis=1)
y = data2['Score']

#Split the data (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#Z-score scaler for size normalization
scaler = StandardScaler()

#Fit the scaler ONLY on the training data, then transform both sets
X_train[['Series_Size']] = scaler.fit_transform(X_train[['Series_Size']])

#Use .transform() on the test set - do NOT fit it again!
X_test[['Series_Size']] = scaler.transform(X_test[['Series_Size']])

In [ ]:
#Drop the leftover text columns from both sets
columns_to_drop = ['Name', 'Producers']
X_train = X_train.drop(columns_to_drop, axis=1, errors='ignore')
X_test = X_test.drop(columns_to_drop, axis=1, errors='ignore')

#Calculate correlation
correlations = X_train.corrwith(y_train).sort_values(ascending=False)

print("\nTop 5 Positive Predictors")
print(correlations.head(5))

print("\nTop 5 Negative Predictors")
print(correlations.tail(5))

Remaining text columns: Index([], dtype='object')


/usr/local/lib/python3.13/dist-packages/numpy/lib/_function_base_impl.py:2999: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.13/dist-packages/numpy/lib/_function_base_impl.py:3000: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]



--- Top 5 Positive Predictors ---
Type_TV                         0.171340
Source_Manga                    0.162098
Rating_R                        0.145460
Suspense                        0.137694
Licensors_Aniplex of America    0.134106
dtype: float64

--- Top 5 Negative Predictors ---
Rating_R+                       -0.211010
Ecchi                           -0.213461
Type_OVA                        -0.233747
Source_Web novel                      NaN
Studios_Bandai Namco Pictures         NaN
dtype: float64


Initial Test

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
import numpy as np

#Initialize model
lreg_model = LinearRegression()

#Train model
lreg_model.fit(X_train, y_train)

#Get predicitons
y_predictions = lreg_model.predict(X_test)

#Calculate all four evaluation metrics
mae = mean_absolute_error(y_test, y_predictions)
r2 = r2_score(y_test, y_predictions)
mse = mean_squared_error(y_test, y_predictions)
rmse = np.sqrt(mse)

print('MSE:', mse )
print('MAE:', mae)
print('RMSE:', rmse )
print('R2:', r2)

MSE: 0.3639742783291214
MAE: 0.4702472901738149
RMSE: 0.6033028081561708
R2: 0.3422208582680937
